In [8]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import wandb

# Initialize wandb
wandb.init(
    project="resnet-real-vs-veo3",
    config={
        "epochs": 10,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "Adam",
        "architecture": "resnet18",
        "img_size": 224,
        "split": [0.7, 0.15, 0.15]
    }
)
config = wandb.config

# Device
device = torch.device("cuda")

# Transforms
transform = transforms.Compose([
    transforms.Resize((config.img_size, config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Dataset
dataset = datasets.ImageFolder(root='output2', transform=transform)
print("Class mapping:", dataset.class_to_idx)

# Train/Val/Test split
total_size = len(dataset)
train_size = int(config.split[0] * total_size)
val_size = int(config.split[1] * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

# Model
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.lr)

# Training loop
for epoch in range(config.epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * train_correct / total
    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100.0 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)

    # Log to wandb
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_accuracy": train_acc,
        "val_loss": avg_val_loss,
        "val_accuracy": val_acc
    })

    print(f"Epoch {epoch+1}: "
          f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")

# Save model
torch.save(model.state_dict(), "resnet_model.pth")
wandb.save("resnet_model.pth")

# Final test evaluation
def evaluate_on_test(model, test_loader):
    model.eval()
    test_correct = 0
    test_total = 0
    test_loss = 0.0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = 100.0 * test_correct / test_total
    avg_test_loss = test_loss / len(test_loader)

    print(f"\n🧪 Final Test Accuracy: {test_acc:.2f}%, Test Loss: {avg_test_loss:.4f}")
    wandb.log({
        "test_accuracy": test_acc,
        "test_loss": avg_test_loss
    })

# Run test evaluation
evaluate_on_test(model, test_loader)


Class mapping: {'real_video_spectra_output': 0, 'veo3_spectra_output': 1}


/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1: Train Loss: 0.4710, Train Acc: 77.27%, Val Loss: 0.5858, Val Acc: 77.78%
Epoch 2: Train Loss: 0.0546, Train Acc: 100.00%, Val Loss: 0.4820, Val Acc: 88.89%
Epoch 3: Train Loss: 0.0326, Train Acc: 100.00%, Val Loss: 0.4038, Val Acc: 100.00%
Epoch 4: Train Loss: 0.0115, Train Acc: 100.00%, Val Loss: 0.3906, Val Acc: 88.89%
Epoch 5: Train Loss: 0.0059, Train Acc: 100.00%, Val Loss: 0.4315, Val Acc: 77.78%
Epoch 6: Train Loss: 0.0031, Train Acc: 100.00%, Val Loss: 0.4353, Val Acc: 77.78%
Epoch 7: Train Loss: 0.0021, Train Acc: 100.00%, Val Loss: 0.4476, Val Acc: 77.78%
Epoch 8: Train Loss: 0.0027, Train Acc: 100.00%, Val Loss: 0.4512, Val Acc: 77.78%
Epoch 9: Train Loss: 0.0009, Train Acc: 100.00%, Val Loss: 0.3905, Val Acc: 88.89%
Epoch 10: Train Loss: 0.0224, Train Acc: 100.00%, Val Loss: 0.2495, Val Acc: 88.89%

🧪 Final Test Accuracy: 100.00%, Test Loss: 0.0900


In [9]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import wandb

# Initialize wandb
wandb.init(
    project="resnet-real-vs-sora",
    config={
        "epochs": 10,
        "batch_size": 32,
        "lr": 1e-4,
        "optimizer": "Adam",
        "architecture": "resnet18",
        "img_size": 224,
        "split": [0.7, 0.15, 0.15]
    }
)
config = wandb.config

# Device
device = torch.device("cuda")

# Transforms
transform = transforms.Compose([
    transforms.Resize((config.img_size, config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Dataset
dataset = datasets.ImageFolder(root='sora_output', transform=transform)
print("Class mapping:", dataset.class_to_idx)

# Train/Val/Test split
total_size = len(dataset)
train_size = int(config.split[0] * total_size)
val_size = int(config.split[1] * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False)

# Model
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=config.lr)

# Training loop
for epoch in range(config.epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100.0 * train_correct / total
    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100.0 * val_correct / val_total
    avg_val_loss = val_loss / len(val_loader)

    # Log to wandb
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_accuracy": train_acc,
        "val_loss": avg_val_loss,
        "val_accuracy": val_acc
    })

    print(f"Epoch {epoch+1}: "
          f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")

# Save model
torch.save(model.state_dict(), "resnet_model.pth")
wandb.save("resnet_model.pth")

# Final test evaluation
def evaluate_on_test(model, test_loader):
    model.eval()
    test_correct = 0
    test_total = 0
    test_loss = 0.0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)

    test_acc = 100.0 * test_correct / test_total
    avg_test_loss = test_loss / len(test_loader)

    print(f"\n🧪 Final Test Accuracy: {test_acc:.2f}%, Test Loss: {avg_test_loss:.4f}")
    wandb.log({
        "test_accuracy": test_acc,
        "test_loss": avg_test_loss
    })

# Run test evaluation
evaluate_on_test(model, test_loader)

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁
test_loss,▁
train_accuracy,▁█████████
train_loss,█▂▁▁▁▁▁▁▁▁
val_accuracy,▁▄█▄▁▁▁▁▄▄
val_loss,█▆▄▄▅▅▅▅▄▁
epoch,10
test_accuracy,100
test_loss,0.09
train_accuracy,100


Class mapping: {'real_video_spectra_output': 0, 'sora_spectra_output': 1}


/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/aarushi/Detecting-Deepfakes/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1: Train Loss: 1.0830, Train Acc: 39.13%, Val Loss: 0.7620, Val Acc: 60.00%
Epoch 2: Train Loss: 0.2996, Train Acc: 91.30%, Val Loss: 0.7165, Val Acc: 60.00%
Epoch 3: Train Loss: 0.0758, Train Acc: 100.00%, Val Loss: 0.7106, Val Acc: 60.00%
Epoch 4: Train Loss: 0.0272, Train Acc: 100.00%, Val Loss: 0.7132, Val Acc: 60.00%
Epoch 5: Train Loss: 0.0141, Train Acc: 100.00%, Val Loss: 0.6789, Val Acc: 60.00%
Epoch 6: Train Loss: 0.0250, Train Acc: 100.00%, Val Loss: 0.5673, Val Acc: 70.00%
Epoch 7: Train Loss: 0.0068, Train Acc: 100.00%, Val Loss: 0.4602, Val Acc: 70.00%
Epoch 8: Train Loss: 0.0089, Train Acc: 100.00%, Val Loss: 0.3831, Val Acc: 80.00%
Epoch 9: Train Loss: 0.0060, Train Acc: 100.00%, Val Loss: 0.3119, Val Acc: 80.00%
Epoch 10: Train Loss: 0.0041, Train Acc: 100.00%, Val Loss: 0.2923, Val Acc: 80.00%

🧪 Final Test Accuracy: 100.00%, Test Loss: 0.0895
